# 🍳 AI Cooking Assistant
### MLH Global Hack Week — Season Kickoff

A personalised meal suggester that learns your preferences over time.

**What we're building:**
1. An onboarding flow that builds your cooking profile
2. A meal suggester that reads your profile and asks Gemini for ideas
3. A feedback loop that rates suggestions and improves over time

**Stack:** Python · Pandas · Gemini API · Gradio

---

> **Running on Google Colab?** Add your Gemini API key as a secret:
> 1. Click the 🔑 key icon in the left sidebar
> 2. Add a secret named `GEMINI_API_KEY`
> 3. Paste your key from [aistudio.google.com](https://aistudio.google.com) (free, no credit card)
> 4. Enable notebook access for the secret


## 0. Setup

Install dependencies and configure your free Gemini API key.

Get yours free at [aistudio.google.com](https://aistudio.google.com) — sign in with Google, click **Get API key**. No credit card needed.

In [ ]:
!pip install gradio google-generativeai pandas -q

In [ ]:
import gradio as gr
import google.generativeai as genai
import pandas as pd
import os
from pathlib import Path

# --- API Key Setup ---
# On Google Colab: uses the secrets manager (recommended)
# Locally: paste your key directly into the string below

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
    print("✅ Loaded API key from Colab secrets")
except Exception:
    API_KEY = ""  # TODO: paste your key here if running locally
    print("Running locally — paste your API key above")

if not API_KEY:
    print("⚠️  No API key found. Add it to Colab secrets or paste it above.")
else:
    genai.configure(api_key=API_KEY)
    model = genai.GenerativeModel("gemini-1.5-flash")
    print("✅ Gemini ready")

## 1. The Profile CSV

Before we build the UI, let's think about what we're storing.

Every user has a profile that tracks:
- Their kitchen equipment
- Dietary restrictions and allergies
- Time they typically have available
- Flavour preferences
- Past meal suggestions and how they rated them

This is what makes suggestions get better over time — the prompt we send Gemini gets richer with every session.

> **Note on Colab:** the CSV lives in your Colab session storage. It resets when your session ends — that's fine for this workshop. In a real app you'd save to Google Drive or a database.


In [ ]:
# Profile file locations
PROFILE_PATH = Path("data/profile.csv")
HISTORY_PATH = Path("data/history.csv")

# Create data directory if it doesn't exist
PROFILE_PATH.parent.mkdir(exist_ok=True)

def load_profile():
    """Load the user profile, return empty dict if it doesn't exist yet."""
    if PROFILE_PATH.exists():
        df = pd.read_csv(PROFILE_PATH)
        return df.iloc[0].to_dict() if len(df) > 0 else {}
    return {}

def save_profile(profile: dict):
    """Save the user profile to CSV."""
    pd.DataFrame([profile]).to_csv(PROFILE_PATH, index=False)
    print(f"Profile saved ✅")

def load_history():
    """Load past meal suggestions and ratings."""
    if HISTORY_PATH.exists():
        return pd.read_csv(HISTORY_PATH)
    return pd.DataFrame(columns=["meal", "rating", "date"])

def save_to_history(meal: str, rating: int):
    """Append a rated meal to history."""
    history = load_history()
    new_row = pd.DataFrame([{
        "meal": meal,
        "rating": rating,
        "date": pd.Timestamp.now().strftime("%Y-%m-%d")
    }])
    history = pd.concat([history, new_row], ignore_index=True)
    history.to_csv(HISTORY_PATH, index=False)
    print(f"Saved to history: {meal[:50]}... ({rating}/5)")

print("Profile functions ready ✅")
print(f"Existing profile: {load_profile()}")

## 2. Onboarding UI

The first time someone uses the app, they answer a series of questions. This gets saved to their profile CSV and used to build smarter prompts.

Run this cell to launch the onboarding form, fill it in, and save your profile.


In [ ]:
def save_onboarding(equipment, dietary, allergies, time_available, spice_level, cuisine_prefs):
    """Save the onboarding answers to the profile CSV."""
    profile = {
        "equipment": ", ".join(equipment) if equipment else "hob",
        "dietary": ", ".join(dietary) if dietary else "none",
        "allergies": allergies or "none",
        "time_available": time_available,
        "spice_level": spice_level,
        "cuisine_prefs": ", ".join(cuisine_prefs) if cuisine_prefs else "varied",
    }
    save_profile(profile)
    return f"✅ Profile saved! Equipment: {profile['equipment']}. Time: {profile['time_available']} mins. Let's cook!"

with gr.Blocks(title="AI Cooking Assistant — Setup") as onboarding_ui:
    gr.Markdown("## 👋 Let's set up your cooking profile")
    gr.Markdown("Answer these once and we'll personalise every suggestion to your kitchen.")

    equipment = gr.CheckboxGroup(
        choices=["Hob", "Oven", "Air fryer", "Microwave", "Instant Pot", "BBQ"],
        label="What cooking equipment do you have?",
        value=["Hob", "Oven"]
    )

    dietary = gr.CheckboxGroup(
        choices=["None", "Vegetarian", "Vegan", "Pescatarian", "Halal", "Kosher"],
        label="Dietary requirements",
        value=["None"]
    )

    allergies = gr.Textbox(
        label="Any allergies? (e.g. nuts, dairy, gluten)",
        placeholder="Leave blank if none",
    )

    time_available = gr.Slider(
        minimum=10, maximum=120, value=30, step=5,
        label="How many minutes do you usually have to cook?"
    )

    spice_level = gr.Radio(
        choices=["Mild", "Medium", "Hot", "Extra hot"],
        label="Spice preference",
        value="Medium"
    )

    cuisine_prefs = gr.CheckboxGroup(
        choices=["Italian", "Asian", "Mexican", "Indian", "Middle Eastern", "British", "American", "Mediterranean"],
        label="Favourite cuisines",
        value=["Italian", "Asian"]
    )

    save_btn = gr.Button("Save my profile →", variant="primary")
    output = gr.Textbox(label="Status")

    save_btn.click(
        fn=save_onboarding,
        inputs=[equipment, dietary, allergies, time_available, spice_level, cuisine_prefs],
        outputs=output
    )

# share=True is required on Colab — creates a temporary public URL
onboarding_ui.launch(share=True)

## 3. Building the Prompt

This is the most interesting part — how you structure the prompt determines the quality of the suggestion.

We build a prompt that includes:
- The user's full profile
- Their past highly-rated meals (so Gemini knows what they like)
- Meals they didn't enjoy (so Gemini avoids them)
- What they have available tonight

**This is prompt engineering** — the context you give the LLM matters as much as the question itself. Run the cell below and read what gets sent to Gemini.


In [ ]:
def build_prompt(tonight_info: str, profile: dict, history: pd.DataFrame) -> str:
    """
    Build a rich prompt using the user's profile and history.
    The more data we have, the better this gets.
    """
    if len(history) > 0:
        top_meals = history[history["rating"] >= 4]["meal"].tolist()
        avoided_meals = history[history["rating"] <= 2]["meal"].tolist()
    else:
        top_meals = []
        avoided_meals = []

    prompt = f"""You are a personal cooking assistant. Suggest ONE specific meal for tonight based on this person's profile.

THEIR COOKING SETUP:
- Equipment available: {profile.get('equipment', 'hob and oven')}
- Dietary requirements: {profile.get('dietary', 'none')}
- Allergies: {profile.get('allergies', 'none')}
- Typical time available: {profile.get('time_available', 30)} minutes
- Spice preference: {profile.get('spice_level', 'medium')}
- Favourite cuisines: {profile.get('cuisine_prefs', 'varied')}

WHAT THEY'VE ENJOYED BEFORE:
{', '.join(top_meals) if top_meals else 'No history yet — suggest something crowd-pleasing'}

MEALS THEY DIDN\'T ENJOY:
{', '.join(avoided_meals) if avoided_meals else 'None recorded'}

TONIGHT\'S SITUATION:
{tonight_info}

Respond with:
1. The meal name
2. Why it suits them specifically (reference their profile)
3. Key ingredients needed
4. Rough time to cook
5. One sentence on how to start

Be specific and practical. One suggestion only."""

    return prompt

# Test it with a dummy profile so you can see the prompt
test_profile = {
    "equipment": "Hob, Oven",
    "dietary": "None",
    "allergies": "nuts",
    "time_available": 30,
    "spice_level": "Medium",
    "cuisine_prefs": "Italian, Asian"
}

print("=== PROMPT SENT TO GEMINI ===\n")
print(build_prompt(
    "I have chicken, some pasta, and about 35 minutes",
    test_profile,
    load_history()
))

## 4. The Main App

Now we wire everything together — the user describes their situation tonight, we build the prompt from their profile, call Gemini, and show the suggestion.

They can then rate it, which gets saved to history and improves future prompts.


In [ ]:
# Store the last suggestion so we can rate it
last_suggestion = {"text": ""}

def get_suggestion(tonight_info: str):
    """Get a meal suggestion from Gemini based on profile and tonight's situation."""
    if not tonight_info.strip():
        return "Tell me what you've got — ingredients, time, how hungry you are!", gr.update(visible=False)

    profile = load_profile()
    if not profile:
        return "⚠️ No profile found — run the onboarding section above first!", gr.update(visible=False)

    history = load_history()
    prompt = build_prompt(tonight_info, profile, history)

    try:
        response = model.generate_content(prompt)
        suggestion = response.text
        last_suggestion["text"] = suggestion
        return suggestion, gr.update(visible=True)
    except Exception as e:
        return f"Error calling Gemini: {e}", gr.update(visible=False)

def rate_suggestion(rating: int):
    """Save the rating for the last suggestion."""
    if not last_suggestion["text"]:
        return "No suggestion to rate yet!"
    save_to_history(last_suggestion["text"][:100], rating)
    return f"Thanks! Rated {rating}/5 — I'll remember that for next time 🙏"

with gr.Blocks(title="AI Cooking Assistant") as main_ui:
    gr.Markdown("## 🍳 What should I cook tonight?")

    tonight = gr.Textbox(
        label="What's your situation tonight?",
        placeholder="e.g. I've got chicken, some leftover rice, and about 40 minutes. Pretty tired so nothing too complicated.",
        lines=3
    )

    suggest_btn = gr.Button("Suggest something →", variant="primary")

    suggestion_output = gr.Textbox(
        label="Tonight's suggestion",
        lines=10,
        interactive=False
    )

    with gr.Column(visible=False) as rating_row:
        gr.Markdown("### How does that sound?")
        with gr.Row():
            rating = gr.Slider(minimum=1, maximum=5, value=3, step=1, label="Rate this suggestion (1-5)")
            rate_btn = gr.Button("Save rating")
        rating_status = gr.Textbox(label="", interactive=False)

    suggest_btn.click(
        fn=get_suggestion,
        inputs=[tonight],
        outputs=[suggestion_output, rating_row]
    )

    rate_btn.click(
        fn=rate_suggestion,
        inputs=[rating],
        outputs=[rating_status]
    )

# share=True required on Colab
main_ui.launch(share=True)

## 5. Your Cooking History

See how your profile is building up over time.


In [ ]:
def show_profile_and_history():
    profile = load_profile()
    history = load_history()

    print("=== YOUR PROFILE ===")
    if profile:
        for key, value in profile.items():
            print(f"  {key}: {value}")
    else:
        print("  No profile yet — run the onboarding section!")

    print(f"\n=== MEAL HISTORY ({len(history)} entries) ===")
    if len(history) > 0:
        print(history.to_string(index=False))
    else:
        print("  No history yet — rate some suggestions!")

show_profile_and_history()

## 6. Mini Challenge (10 mins)

Pick one to extend the app:

1. **Ingredients tracker** — add a fridge inventory to onboarding, update it after each meal suggestion
2. **Weekly planner** — modify the prompt to ask Gemini for 5 meals for the week instead of just tonight
3. **Nutritional goal** — add a dietary goal (high protein, low carb etc) to the profile and include it in the prompt
4. **Cuisine roulette** — add a "Surprise me" button that picks a random cuisine outside their usual preferences and prompts accordingly

Share your Colab link in the chat when you're done! 🚀
